# Fake News Detection with a Fine-Tuned DeBERTa-v3

**7PAM2015 Research Methods – Assignment 3**

The task is binary text classification: given a news article (headline + body), decide whether it is **fake (0)** or **real (1)**. I fine-tune `microsoft/deberta-v3-base` and compare it against three simpler baselines, so I can see how much the transformer *and* the fine-tuning actually add over cheaper methods.

**Dataset:** [`GonzaloA/fake_news`](https://huggingface.co/datasets/GonzaloA/fake_news) – about 40k English news articles with ready-made train / validation / test splits.

**Pipeline:** load data → clean + leakage check → baselines (TF-IDF + LogReg / NB, frozen DeBERTa) → fine-tune DeBERTa with a custom head → evaluate (accuracy, precision, recall, F1, ROC-AUC) → error analysis → save the model + a small Gradio demo.

> Set the Colab runtime to a **T4 GPU** before running (`Runtime → Change runtime type`). A full run takes roughly an hour.

### **Install packages**

Colab already ships torch, scikit-learn, pandas and numpy. These are the extras I need: Hugging Face `transformers`/`datasets`, the SentencePiece backend the DeBERTa tokenizer uses, Plotly for the charts, and Gradio for the demo at the end.

In [1]:
# install the libraries Colab doesn't already have
!pip install -q "transformers>=4.41" "datasets>=2.19" sentencepiece protobuf "plotly>=5.20" gradio

### **Imports, config and reproducibility**

All imports in one place, a `set_seed` helper, and a small `CFG` dataclass holding every hyperparameter. Keeping the settings in one object means I can re-run the whole experiment with different values by editing one place (the LR search later just calls `replace` on it). I also select the GPU here.

In [2]:
import os, re, gc, math, time, random, warnings
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve,
                             precision_recall_curve, confusion_matrix)

from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModel, AutoConfig,
                          DataCollatorWithPadding,
                          get_linear_schedule_with_warmup)
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


@dataclass
class CFG:
    model_name: str    = "microsoft/deberta-v3-base"
    max_len: int       = 256
    batch_size: int    = 16
    epochs: int        = 2
    encoder_lr: float  = 2e-5
    head_lr: float     = 1e-4
    llrd: float        = 0.9
    weight_decay: float = 0.01
    warmup_ratio: float = 0.1
    n_msd: int         = 5
    dropout: float     = 0.3
    grad_clip: float   = 1.0
    seed: int          = 42
    num_labels: int    = 2
    fast_mode: bool    = False

cfg = CFG()
set_seed(cfg.seed)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type != "cuda":
    print("WARNING: no GPU detected. Runtime -> Change runtime type -> T4 GPU.")

Device: cuda


### **Load the dataset**

Pull the dataset from the Hub and turn each split into a pandas DataFrame for easy exploration. I drop the leftover index column and tag every row with the split it came from, then concatenate so I can do EDA across all of it.

In [3]:
raw = load_dataset("GonzaloA/fake_news")

def to_frame(split):
    df = raw[split].to_pandas()
    df = df.drop(columns=[c for c in df.columns if c.lower().startswith("unnamed")])
    df["split"] = split
    return df

df_all = pd.concat([to_frame(s) for s in ["train", "validation", "test"]],
                   ignore_index=True)
print(df_all.shape)
df_all.head(3)

README.md:   0%|          | 0.00/6.73k [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


dataset_infos.json:   0%|          | 0.00/1.03k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 38.8MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

Generating train split:   0%|          | 0/24353 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/8117 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/8117 [00:00<?, ? examples/s]

(40587, 4)


,title,text,label,split
0,‘Maury’ Show Official Facebook Posts F*CKED U...,Maury is perhaps one of the trashiest shows on...,0,train
1,Trump’s Favorite News Channel Tries To Soothe...,"Yesterday, after the father of one of the UCLA...",0,train
2,"Russia warns Iraq, Kurds not to destabilize Mi...",MOSCOW (Reuters) - Russia on Wednesday warned ...,1,train


### **Cleaning and the leakage check**

Two things here. First, basic hygiene: drop empty/tiny rows and exact duplicate articles within each split. Second, and more important, a **leakage check** – if the same article appears in both train and val/test, the model is effectively tested on something it memorised, so I remove any val/test article whose text also occurs in train. This is the difference between honest scores and inflated ones.

In [4]:
df_all["title"] = df_all["title"].fillna("").str.strip()
df_all["text"]  = df_all["text"].fillna("").str.strip()
df_all = df_all[df_all["text"].str.len() > 10].reset_index(drop=True)
before = len(df_all)
df_all = df_all.drop_duplicates(subset=["split", "text"]).reset_index(drop=True)
print(f"Removed {before - len(df_all)} within-split duplicate articles.")
train_texts = set(df_all.loc[df_all.split == "train", "text"])
mask_leak = (df_all.split != "train") & (df_all.text.isin(train_texts))
print(f"Removed {mask_leak.sum()} leaked articles from validation/test.")
df_all = df_all[~mask_leak].reset_index(drop=True)

train_df = df_all[df_all.split == "train"].reset_index(drop=True)
valid_df = df_all[df_all.split == "validation"].reset_index(drop=True)
test_df  = df_all[df_all.split == "test"].reset_index(drop=True)

if cfg.fast_mode:
    train_df = (train_df.groupby("label", group_keys=False)
                        .apply(lambda g: g.sample(frac=0.2, random_state=cfg.seed))
                        .reset_index(drop=True))
    print("fast_mode ON -> training on", len(train_df), "articles")

print({s: len(d) for s, d in [("train", train_df), ("valid", valid_df), ("test", test_df)]})

Removed 1 within-split duplicate articles.
Removed 3 leaked articles from validation/test.
{'train': 24342, 'valid': 8109, 'test': 8110}


### **EDA 1 – class balance**

A grouped bar chart of fake vs real counts per split. The classes are close to balanced, which is why accuracy is an acceptable headline metric here – though I still report precision/recall/F1 later regardless.

In [5]:
counts = df_all.groupby(["split", "label"]).size().rename("n").reset_index()
counts["label"] = counts["label"].map({0: "fake", 1: "real"})

fig = px.bar(counts, x="split", y="n", color="label", barmode="group",
             category_orders={"split": ["train", "validation", "test"]},
             color_discrete_map={"fake": "#EF553B", "real": "#636EFA"},
             title="Class balance per split", labels={"n": "articles"})
fig.update_layout(template="plotly_white", width=700, height=400)
fig.show()

### **EDA 2 – article length**

Distribution of article length (in words) by class. Two reasons to look: it informs the `max_len` choice, and it's a sanity check that length on its own isn't a giveaway for the label.

In [6]:
df_all["n_words"] = df_all["text"].str.split().str.len()

sampled = df_all.sample(min(len(df_all), 20000), random_state=cfg.seed).copy()
sampled["class"] = sampled["label"].map({0: "fake", 1: "real"})
fig = px.histogram(sampled, x="n_words", color="class", nbins=80, barmode="overlay",
                   opacity=0.6, marginal="box",
                   color_discrete_map={"fake": "#EF553B", "real": "#636EFA"},
                   title="Article length (words) by class",
                   labels={"n_words": "words per article", "color": "label"})
fig.update_layout(template="plotly_white", width=800, height=450)
fig.update_xaxes(range=[0, 1500])
fig.show()

print(df_all.groupby("label")["n_words"].describe().round(1))

         count   mean    std  min    25%    50%    75%     max
label                                                         
0      18640.0  429.2  375.5  1.0  269.0  374.0  505.0  8135.0
1      21921.0  390.2  301.2  2.0  145.0  358.0  526.0  5828.0


### **Text cleaning**

I keep cleaning deliberately light. Transformers are pretrained on raw text, so lowercasing or removing stopwords throws away signal the model can use. I just join title + body, strip URLs and collapse whitespace. Casing and punctuation like "!!" are kept on purpose because they're genuinely informative for fake news.

In [7]:
URL_RE = re.compile(r"https?://\S+|www\.\S+")
WS_RE  = re.compile(r"\s+")

def clean_text(title, body):
    # join title + body, drop urls, squeeze whitespace. keep casing/punctuation
    # on purpose - ALL CAPS and "!!" are useful signals here.
    merged = f"{title} [SEP] {body}"
    merged = URL_RE.sub(" ", merged)
    merged = WS_RE.sub(" ", merged)
    return merged.strip()

for d in (train_df, valid_df, test_df):
    d["input_text"] = [clean_text(t, x) for t, x in zip(d["title"], d["text"])]

train_df["input_text"].iloc[0][:300]

'‘Maury’ Show Official Facebook Posts F*CKED UP Caption On Guest That Looks Like Ted Cruz (IMAGE) [SEP] Maury is perhaps one of the trashiest shows on television today. It s right in line with the likes of the gutter trash that is Jerry Springer, and the fact that those shows are still on the air wit'

### **Tokenizer**

Load the DeBERTa-v3 tokenizer from the same checkpoint as the model, so the vocabulary always matches the weights. It's a SentencePiece tokenizer with a ~128k vocab.

In [8]:
tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)
print(tokenizer)

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

DebertaV2Tokenizer(name_or_path='microsoft/deberta-v3-base', vocab_size=128000, model_max_length=1000000000000000019884624838656, padding_side='right', truncation_side='right', special_tokens={'bos_token': '[CLS]', 'eos_token': '[SEP]', 'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128000: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})


### **Choosing `max_len`**

Before fixing the truncation length I look at the real token counts on a sample. The median is around 450 tokens, so 256 doesn't fully cover most articles – but attention cost grows quadratically with sequence length, and news writing front-loads the key facts, so 256 is a sensible trade-off. The final F1 confirms the opening is enough; I flag truncation as a limitation in the report.

In [9]:
sample_tok = [len(tokenizer.encode(t, truncation=False))
              for t in train_df["input_text"].sample(2000, random_state=cfg.seed)]
pct = np.percentile(sample_tok, [50, 75, 90, 95])
print(f"token-count percentiles  50/75/90/95%: {pct.round(0)}")
print(f"share of articles fully covered by max_len={cfg.max_len}: "
      f"{np.mean(np.array(sample_tok) <= cfg.max_len):.1%}")

fig = px.histogram(x=sample_tok, nbins=60,
                   title="Sub-word tokens per article (2,000-article sample)",
                   labels={"x": "tokens"})
fig.add_vline(x=cfg.max_len, line_dash="dash", line_color="red",
              annotation_text=f"max_len={cfg.max_len}")
fig.update_layout(template="plotly_white", width=750, height=400)
fig.show()

token-count percentiles  50/75/90/95%: [ 458.  644.  926. 1142.]
share of articles fully covered by max_len=256: 25.2%


### **Dataset and dataloaders**

A small `Dataset` that tokenizes one article at a time, plus a collator that pads each batch only to its own longest sequence (dynamic padding, which is cheaper than always padding to 256). A single `make_loader` helper builds all three loaders with identical settings.

In [10]:
class NewsDataset(torch.utils.data.Dataset):
    def __init__(self, frame):
        self.texts  = frame["input_text"].tolist()
        self.labels = frame["label"].astype(int).tolist()
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        enc = tokenizer(self.texts[idx], truncation=True, max_length=cfg.max_len)
        enc["labels"] = self.labels[idx]
        return enc

collate = DataCollatorWithPadding(tokenizer=tokenizer)   # pad per-batch, not to a fixed 256

def make_loader(frame, shuffle):
    return DataLoader(NewsDataset(frame), batch_size=cfg.batch_size, shuffle=shuffle,
                      collate_fn=collate, num_workers=2, pin_memory=True)

train_loader = make_loader(train_df, shuffle=True)
valid_loader = make_loader(valid_df, shuffle=False)
test_loader  = make_loader(test_df,  shuffle=False)
print(f"{len(train_loader)} training batches of {cfg.batch_size}")

1522 training batches of 16


### **Metrics helper**

One function turns predicted probabilities into the full metric set and stores everything in `RESULTS` / `PROBS`. Every model below is scored through this same function, so the comparison is fair and I can build the comparison charts later without re-running anything.

In [11]:
RESULTS = {}
PROBS   = {}

def evaluate_model(name, y_true, y_prob):
    y_pred = (np.asarray(y_prob) >= 0.5).astype(int)
    metrics = {
        "accuracy":  accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "recall":    recall_score(y_true, y_pred),
        "f1":        f1_score(y_true, y_pred),
        "roc_auc":   roc_auc_score(y_true, y_prob),
    }
    RESULTS[name] = metrics
    PROBS[name] = np.asarray(y_prob)
    print(name, {k: round(v, 4) for k, v in metrics.items()})
    return metrics

y_test = test_df["label"].values

### **Baselines 1 & 2 – TF-IDF + Logistic Regression / Naive Bayes**

Two classic text-classification baselines on TF-IDF features (uni- and bi-grams, 50k terms). These are meant to be strong rather than strawmen: if the fine-tuned model can't beat them, it isn't worth the extra compute.

In [12]:
tfidf = TfidfVectorizer(max_features=50_000, ngram_range=(1, 2),
                        sublinear_tf=True, strip_accents="unicode")
X_train = tfidf.fit_transform(train_df["input_text"])
X_test  = tfidf.transform(test_df["input_text"])

logreg = LogisticRegression(max_iter=2000, C=1.0, n_jobs=-1)
logreg.fit(X_train, train_df["label"])
evaluate_model("TF-IDF + LogReg", y_test, logreg.predict_proba(X_test)[:, 1])

nb = MultinomialNB()
nb.fit(X_train, train_df["label"])
evaluate_model("TF-IDF + NaiveBayes", y_test, nb.predict_proba(X_test)[:, 1])

TF-IDF + LogReg {'accuracy': 0.9744, 'precision': 0.9809, 'recall': 0.9709, 'f1': 0.9759, 'roc_auc': np.float64(0.9946)}
TF-IDF + NaiveBayes {'accuracy': 0.9461, 'precision': 0.9543, 'recall': 0.9444, 'f1': 0.9493, 'roc_auc': np.float64(0.9816)}


{'accuracy': 0.9461159062885327,
 'precision': 0.9542803825519011,
 'recall': 0.9443674976915974,
 'f1': 0.9492980624202344,
 'roc_auc': np.float64(0.9816033511797662)}

### **Baseline 3 – frozen DeBERTa + Logistic Regression**

This is the key control. I push articles through DeBERTa with **no fine-tuning**, mean-pool the token vectors into one document embedding, and train logistic regression on top. Comparing this against the fine-tuned model isolates how much the fine-tuning itself adds, separate from the pretrained representation.

In [13]:
@torch.no_grad()
def embed_texts(texts, batch_size=64, desc="embedding"):
    encoder = AutoModel.from_pretrained(cfg.model_name).float().to(DEVICE).eval()
    out = []
    for i in tqdm(range(0, len(texts), batch_size), desc=desc):
        enc = tokenizer(list(texts[i:i + batch_size]), truncation=True,
                        max_length=cfg.max_len, padding=True,
                        return_tensors="pt").to(DEVICE)
        h = encoder(**enc).last_hidden_state
        mask = enc["attention_mask"].unsqueeze(-1)
        pooled = (h * mask).sum(1) / mask.sum(1)
        out.append(pooled.cpu().numpy())
    del encoder; gc.collect(); torch.cuda.empty_cache()
    return np.vstack(out)

probe_idx = train_df.sample(min(6000, len(train_df)), random_state=cfg.seed).index
E_train = embed_texts(train_df.loc[probe_idx, "input_text"].values, desc="train embeds")
E_test  = embed_texts(test_df["input_text"].values, desc="test embeds")

frozen_lr = LogisticRegression(max_iter=2000)
frozen_lr.fit(E_train, train_df.loc[probe_idx, "label"])
evaluate_model("Frozen DeBERTa + LogReg", y_test, frozen_lr.predict_proba(E_test)[:, 1])

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  371MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


train embeds:   0%|          | 0/94 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


test embeds:   0%|          | 0/127 [00:00<?, ?it/s]

Frozen DeBERTa + LogReg {'accuracy': 0.9803, 'precision': 0.9824, 'recall': 0.9806, 'f1': 0.9815, 'roc_auc': np.float64(0.9989)}


{'accuracy': 0.9802712700369913,
 'precision': 0.9824236817761333,
 'recall': 0.9806094182825484,
 'f1': 0.9815157116451017,
 'roc_auc': np.float64(0.9989133155113411)}

### **Attention pooling**

The first part of my custom head. Instead of using the `[CLS]` token, this learns a weight for every token and takes the weighted average, so the model decides which tokens matter for this task. The weights are also useful for a bit of explainability in the demo. Padded positions are masked out before the softmax.

In [14]:
class AttentionPooling(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, 1),
        )
    def forward(self, hidden_states, attention_mask):
        scores = self.attn(hidden_states).squeeze(-1)
        scores = scores.masked_fill(attention_mask == 0, -1e4)
        weights = torch.softmax(scores, dim=-1).unsqueeze(-1)
        return (weights * hidden_states).sum(dim=1), weights.squeeze(-1)

### **The classifier**

Everything together: DeBERTa backbone → attention pooling → multi-sample dropout → linear layer. Multi-sample dropout runs the pooled vector through several dropout masks and averages the logits, a cheap ensemble-style trick that steadies fine-tuning. The `.float()` on the backbone forces fp32 weights, which avoids an fp16/AMP clash during training.

In [15]:
class FakeNewsClassifier(nn.Module):
    def __init__(self, model_name, num_labels, n_msd, p_drop):
        super().__init__()
        self.config   = AutoConfig.from_pretrained(model_name)
        self.backbone = AutoModel.from_pretrained(model_name).float()
        self.pool     = AttentionPooling(self.config.hidden_size)
        self.dropouts = nn.ModuleList(nn.Dropout(p_drop) for _ in range(n_msd))
        self.classifier = nn.Linear(self.config.hidden_size, num_labels)
        nn.init.normal_(self.classifier.weight, std=0.02)
        nn.init.zeros_(self.classifier.bias)

    def forward(self, input_ids, attention_mask, token_type_ids=None, return_attn=False):
        h = self.backbone(input_ids=input_ids,
                          attention_mask=attention_mask).last_hidden_state
        doc, attn_weights = self.pool(h, attention_mask)
        logits = torch.stack([self.classifier(d(doc)) for d in self.dropouts]).mean(0)
        return (logits, attn_weights) if return_attn else logits

### **Optimizer groups – layer-wise LR decay**

Rather than one learning rate for everything, the head gets the largest LR and each encoder layer gets a progressively smaller one going down the stack, since the lower layers hold general language knowledge I don't want to disturb much. Bias and LayerNorm parameters are excluded from weight decay, which is standard practice.

In [16]:
def get_optimizer_params(model, encoder_lr, head_lr, weight_decay, llrd):
    no_decay = ("bias", "LayerNorm.weight")
    n_layers = model.config.num_hidden_layers
    groups = []

    def add(params, lr):
        decay   = [p for n, p in params if not any(nd in n for nd in no_decay)]
        nodecay = [p for n, p in params if any(nd in n for nd in no_decay)]
        if decay:   groups.append({"params": decay,   "lr": lr, "weight_decay": weight_decay})
        if nodecay: groups.append({"params": nodecay, "lr": lr, "weight_decay": 0.0})


    add([(n, p) for n, p in model.named_parameters() if "backbone" not in n], head_lr)
    add([(n, p) for n, p in model.named_parameters() if "backbone.embeddings" in n],
        encoder_lr * llrd ** n_layers)
    for k in range(n_layers):
        add([(n, p) for n, p in model.named_parameters()
             if f"backbone.encoder.layer.{k}." in n],
            encoder_lr * llrd ** (n_layers - 1 - k))
    seen = {f"backbone.encoder.layer.{k}." for k in range(n_layers)}
    add([(n, p) for n, p in model.named_parameters()
         if "backbone" in n and "backbone.embeddings" not in n
         and not any(s in n for s in seen)], encoder_lr)
    return groups

### **Training and evaluation loops**

`run_epoch` does one pass over a loader and is shared by both training and evaluation, so the forward path is identical either way (pass an optimizer to train, leave it out to evaluate). `fit` is the full loop: AdamW with the layer-wise groups, a warmup schedule, mixed precision, gradient clipping, and early stopping that keeps the best model by validation F1.

In [17]:
loss_fn = nn.CrossEntropyLoss()

def run_epoch(model, loader, optimizer=None, scheduler=None, scaler=None):
    training = optimizer is not None
    model.train() if training else model.eval()
    losses, all_labels, all_probs = [], [], []

    for batch in tqdm(loader, leave=False, desc="train" if training else "eval"):
        labels = batch.pop("labels").to(DEVICE)
        batch  = {k: v.to(DEVICE) for k, v in batch.items()
                  if k in ("input_ids", "attention_mask")}
        with torch.set_grad_enabled(training):
            with torch.autocast(device_type="cuda", enabled=DEVICE.type == "cuda"):
                logits = model(**batch)
                loss = loss_fn(logits, labels)
        if training:
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
        losses.append(loss.item())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(torch.softmax(logits.float(), -1)[:, 1].detach().cpu().numpy())
    return float(np.mean(losses)), np.array(all_labels), np.array(all_probs)


def fit(cfg, train_loader, valid_loader, verbose=True):
    set_seed(cfg.seed)
    model = FakeNewsClassifier(cfg.model_name, cfg.num_labels, cfg.n_msd, cfg.dropout).to(DEVICE)
    optimizer = torch.optim.AdamW(
        get_optimizer_params(model, cfg.encoder_lr, cfg.head_lr, cfg.weight_decay, cfg.llrd))
    n_steps = len(train_loader) * cfg.epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer, int(n_steps * cfg.warmup_ratio), n_steps)
    scaler = torch.amp.GradScaler(enabled=DEVICE.type == "cuda")

    history, best_f1, best_state, patience = [], -1.0, None, 1
    for epoch in range(cfg.epochs):
        tr_loss, _, _ = run_epoch(model, train_loader, optimizer, scheduler, scaler)
        va_loss, y, p = run_epoch(model, valid_loader)
        va_f1 = f1_score(y, (p >= 0.5).astype(int))
        history.append({"epoch": epoch + 1, "train_loss": tr_loss,
                        "valid_loss": va_loss, "valid_f1": va_f1})
        if verbose:
            print(f"epoch {epoch+1}: train_loss={tr_loss:.4f} "
                  f"valid_loss={va_loss:.4f} valid_F1={va_f1:.4f}")
        if va_f1 > best_f1:
            best_f1, patience = va_f1, 1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience -= 1
            if patience < 0:
                print("early stopping"); break
    model.load_state_dict(best_state)
    return model, pd.DataFrame(history), best_f1

### **Learning-rate search**

The encoder LR is the setting most likely to make or break fine-tuning (too high and the pretrained weights get wrecked, too low and it underfits), so I sweep three values. Each is trained for one epoch on a 15% stratified slice and scored on the full validation set; the test set stays untouched. The best value is used for the real run.

In [18]:
from dataclasses import replace

subset = (train_df.groupby("label", group_keys=False)
                  .apply(lambda g: g.sample(frac=0.15, random_state=cfg.seed))
                  .reset_index(drop=True))
sub_loader = make_loader(subset, shuffle=True)

search_results = []
for lr in (1e-5, 2e-5, 3e-5):
    print(f"\n--- encoder_lr = {lr:.0e} ---")
    trial_cfg = replace(cfg, encoder_lr=lr, epochs=1)
    _, _, f1_val = fit(trial_cfg, sub_loader, valid_loader, verbose=True)
    search_results.append({"encoder_lr": lr, "valid_f1": f1_val})
    gc.collect(); torch.cuda.empty_cache()

search_df = pd.DataFrame(search_results)
best_lr = search_df.loc[search_df.valid_f1.idxmax(), "encoder_lr"]
print(search_df, f"\nbest encoder_lr: {best_lr:.0e}")

fig = px.bar(search_df, x=search_df.encoder_lr.map("{:.0e}".format), y="valid_f1",
             title="Learning-rate search (1 epoch, 15% subset)",
             labels={"x": "encoder learning rate", "valid_f1": "validation F1"},
             text=search_df.valid_f1.round(4))
fig.update_layout(template="plotly_white", width=650, height=400)
fig.update_yaxes(range=[max(0, search_df.valid_f1.min() - 0.02), 1.0])
fig.show()


--- encoder_lr = 1e-05 ---


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


train:   0%|          | 0/229 [00:00<?, ?it/s]

eval:   0%|          | 0/507 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


train:   0%|          | 0/229 [00:00<?, ?it/s]

eval:   0%|          | 0/507 [00:00<?, ?it/s]

epoch 1: train_loss=0.1234 valid_loss=0.0355 valid_F1=0.9824
   encoder_lr  valid_f1
0     0.00001  0.982131
1     0.00002  0.983117
2     0.00003  0.982368 
best encoder_lr: 2e-05


### **Final training run**

Train on the full training set with the chosen LR, then plot the loss and F1 curves to confirm the model is improving and validation is tracking training (i.e. not overfitting).

In [19]:
cfg.encoder_lr = float(best_lr)
model, history, _ = fit(cfg, train_loader, valid_loader)
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_scatter(x=history.epoch, y=history.train_loss, name="train loss", mode="lines+markers")
fig.add_scatter(x=history.epoch, y=history.valid_loss, name="valid loss", mode="lines+markers")
fig.add_scatter(x=history.epoch, y=history.valid_f1, name="valid F1",
                mode="lines+markers", secondary_y=True)
fig.update_layout(template="plotly_white", width=750, height=400, title="Fine-tuning curves")
fig.update_yaxes(title_text="cross-entropy loss", secondary_y=False)
fig.update_yaxes(title_text="F1", secondary_y=True)
fig.show()

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


train:   0%|          | 0/1522 [00:00<?, ?it/s]

eval:   0%|          | 0/507 [00:00<?, ?it/s]

epoch 1: train_loss=0.0751 valid_loss=0.0298 valid_F1=0.9878


train:   0%|          | 0/1522 [00:00<?, ?it/s]

eval:   0%|          | 0/507 [00:00<?, ?it/s]

epoch 2: train_loss=0.0239 valid_loss=0.0291 valid_F1=0.9912


### **Test-set evaluation**

Run the fine-tuned model on the held-out test set and pull all four models into one table so they can be compared directly.

In [20]:
_, y_true, p_test = run_epoch(model, test_loader)
evaluate_model("Fine-tuned DeBERTa-v3", y_true, p_test)

results_df = (pd.DataFrame(RESULTS).T
                .loc[["TF-IDF + NaiveBayes", "TF-IDF + LogReg",
                      "Frozen DeBERTa + LogReg", "Fine-tuned DeBERTa-v3"]].round(4))
results_df

eval:   0%|          | 0/507 [00:00<?, ?it/s]

Fine-tuned DeBERTa-v3 {'accuracy': 0.9894, 'precision': 0.9914, 'recall': 0.9887, 'f1': 0.9901, 'roc_auc': np.float64(0.9997)}


,accuracy,precision,recall,f1,roc_auc
TF-IDF + NaiveBayes,0.9461,0.9543,0.9444,0.9493,0.9816
TF-IDF + LogReg,0.9744,0.9809,0.9709,0.9759,0.9946
Frozen DeBERTa + LogReg,0.9803,0.9824,0.9806,0.9815,0.9989
Fine-tuned DeBERTa-v3,0.9894,0.9914,0.9887,0.9901,0.9997


### **Metric comparison chart**

The same numbers as a grouped bar chart, which makes the gap between the models easier to read at a glance.

In [21]:
plot_df = results_df.reset_index().melt(id_vars="index", var_name="metric", value_name="score")
fig = px.bar(plot_df, x="metric", y="score", color="index", barmode="group",
             title="Model comparison on the held-out test set", labels={"index": "model"})
fig.update_layout(template="plotly_white", width=900, height=450,
                  yaxis_range=[max(0.0, plot_df.score.min() - 0.02), 1.0])
fig.show()

### **Confusion matrix**

Where the fine-tuned model's mistakes actually land. For a fake-news filter the expensive cell is *fake predicted as real* (fake slipping through), so it's worth inspecting directly instead of trusting accuracy alone.

In [22]:
cm = confusion_matrix(y_true, (p_test >= 0.5).astype(int))
fig = px.imshow(cm, text_auto=True, x=["pred fake", "pred real"], y=["true fake", "true real"],
                color_continuous_scale="Blues", title="Confusion matrix - fine-tuned DeBERTa-v3")
fig.update_layout(width=520, height=450, coloraxis_showscale=False)
fig.show()

### **ROC and precision-recall curves**

A threshold-independent view of all four models on one plot. Useful because a single 0.5 cut-off hides how well-separated the classes are; the curves show the full trade-off.

In [23]:
fig = make_subplots(rows=1, cols=2, subplot_titles=("ROC curves", "Precision-Recall curves"))
for name, prob in PROBS.items():
    fpr, tpr, _ = roc_curve(y_test, prob)
    prec, rec, _ = precision_recall_curve(y_test, prob)
    fig.add_scatter(x=fpr, y=tpr, name=name, mode="lines", row=1, col=1)
    fig.add_scatter(x=rec, y=prec, name=name, mode="lines", row=1, col=2, showlegend=False)
fig.add_scatter(x=[0, 1], y=[0, 1], mode="lines", name="chance",
                line=dict(dash="dot", color="grey"), row=1, col=1)
fig.update_xaxes(title_text="false positive rate", row=1, col=1)
fig.update_yaxes(title_text="true positive rate", row=1, col=1)
fig.update_xaxes(title_text="recall", row=1, col=2)
fig.update_yaxes(title_text="precision", row=1, col=2)
fig.update_layout(template="plotly_white", width=950, height=430)
fig.show()

### **Error analysis – confidence and length**

Two diagnostics: how confident the model is when it's right vs wrong, and whether the error rate depends on article length. The confidence plot in particular tells me whether a "route low-confidence cases to a human" threshold would be useful in practice.

In [24]:
err_df = test_df.copy()
err_df["p_real"]  = p_test
err_df["pred"]    = (p_test >= 0.5).astype(int)
err_df["correct"] = err_df["pred"] == err_df["label"]
err_df["confidence"] = np.abs(err_df["p_real"] - 0.5) * 2    # 0 = unsure, 1 = certain

fig = px.histogram(err_df, x="confidence", color="correct", nbins=40, barmode="overlay",
                   opacity=0.65, histnorm="percent",
                   title="Prediction confidence: correct vs incorrect",
                   color_discrete_map={True: "#00CC96", False: "#EF553B"})
fig.update_layout(template="plotly_white", width=750, height=400)
fig.show()

err_df["len_bin"] = pd.cut(err_df["text"].str.split().str.len(),
                           [0, 100, 250, 500, 1000, 10_000],
                           labels=["<100", "100-250", "250-500", "500-1k", ">1k"])
rate = err_df.groupby("len_bin")["correct"].agg(["mean", "size"]).reset_index()
rate["error_rate"] = 1 - rate["mean"]
fig = px.bar(rate, x="len_bin", y="error_rate", text=rate["size"],
             title="Error rate by article length (bar labels = n articles)",
             labels={"len_bin": "article length (words)", "error_rate": "error rate"})
fig.update_layout(template="plotly_white", width=700, height=400)
fig.show()

### **The most confident mistakes**

Sorting the wrong predictions by confidence surfaces the worst failures – the cases where the model was sure and still wrong. Reading these by hand is the fastest way to spot systematic problems, and they often turn out to be mislabelled or genuinely ambiguous articles.

In [25]:
worst = (err_df[~err_df["correct"]].sort_values("confidence", ascending=False)
         .head(10)[["title", "label", "pred", "p_real"]])
worst["label"] = worst["label"].map({0: "fake", 1: "real"})
worst["pred"]  = worst["pred"].map({0: "fake", 1: "real"})
worst.style.format({"p_real": "{:.3f}"})

,title,label,pred,p_real
7875,breaking nypd ready to make arrests in weiner casehillary visited pedophile island at least timesmoney laundering underage sex payforplayproof of inappropriate handling classified information percentfedupcom,real,fake,0.001
6804,what its like to live in kiev after marrying a ukrainian woman,real,fake,0.001
3137,no title,real,fake,0.003
7829,pin drop speech by father of daughter kidnapped and killed by isis i have voted for donald j trump percentfedupcom,real,fake,0.003
7463,why its absolutely worth it to learn game,real,fake,0.003
1205,wow whistleblower tells chilling story of massive voter fraud trump campaign readies lawsuit against fl sec of elections in critical district video percentfedupcom,real,fake,0.004
3444,trump erupts at secret meeting with the press transition in chaos tweets addicting info the knowledge you crave,real,fake,0.004
5992,bayern munich beat augsburg,real,fake,0.005
6970,evil hillary supporters yell fck trumpburn truck of daddy fishing with yr son over of trump bumperstickers video percentfedupcom,real,fake,0.005
5569,yes creationists can be real scientists too,real,fake,0.006


### **Save the model**

Write the weights and tokenizer to disk so the model can be reloaded later (or pushed to the Hub) without retraining. The commented lines show how to load it back.

In [26]:
SAVE_DIR = "fake_news_deberta"
os.makedirs(SAVE_DIR, exist_ok=True)
torch.save(model.state_dict(), f"{SAVE_DIR}/model_state.pt")
tokenizer.save_pretrained(SAVE_DIR)
print("saved:", os.listdir(SAVE_DIR))

saved: ['tokenizer_config.json', 'model_state.pt', 'tokenizer.json']


### **Gradio demo**

A small web app to try the model on any headline + article. It returns the fake/real probabilities and the tokens the attention layer weighted most, which is a lightweight bit of explainability. `share=True` gives a public link that's handy for a screenshot in the report.

In [27]:
import gradio as gr

@torch.no_grad()
def predict_news(title, body):
    model.eval()
    enc = tokenizer(clean_text(title, body), truncation=True,
                    max_length=cfg.max_len, return_tensors="pt").to(DEVICE)
    logits, attn = model(**{k: enc[k] for k in ("input_ids", "attention_mask")},
                         return_attn=True)
    probs = torch.softmax(logits.float(), -1)[0].cpu().numpy()

    tokens  = tokenizer.convert_ids_to_tokens(enc["input_ids"][0])
    weights = attn[0].float().cpu().numpy()
    pairs = [(t.replace("▁", ""), float(w)) for t, w in zip(tokens, weights)
             if t not in tokenizer.all_special_tokens]
    top = sorted(pairs, key=lambda x: -x[1])[:10]
    top_str = ", ".join(f"{t} ({w:.3f})" for t, w in top if t)
    return ({"fake": float(probs[0]), "real": float(probs[1])}, top_str)

demo = gr.Interface(
    fn=predict_news,
    inputs=[gr.Textbox(label="Headline"), gr.Textbox(label="Article text", lines=8)],
    outputs=[gr.Label(label="Prediction"),
             gr.Textbox(label="Most influential tokens (attention pooling)")],
    title="Fake News Detector - fine-tuned DeBERTa-v3",
    description="Paste a headline and article text to get a fake/real prediction, class "
                "probabilities, and the tokens the model focused on.",
    examples=[
        ["Scientists confirm new exoplanet in habitable zone",
         "Astronomers using the James Webb Space Telescope reported the discovery of an "
         "Earth-sized planet orbiting within the habitable zone of a nearby red dwarf star, "
         "according to a peer-reviewed study published Thursday."],
        ["SHOCKING: You won't BELIEVE what this politician did next!!",
         "Insiders reveal the TRUTH they don't want you to know. Share before this gets taken "
         "down! The mainstream media is hiding everything."],
    ],
)
demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bacaf9121690404e8b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
